In [ ]:
#| default_exp loss

# Loss functions

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn.functional as F, torch.nn as nn

In [ ]:
#| export
def mse_loss(preds, target):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    """
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            loss += F.mse_loss(pred, targ, reduction='mean')
        return loss / preds.size(0)
    else:
        return F.mse_loss(preds, target, reduction='mean')

In [ ]:
#| export
class FocalLoss(nn.Module):
    """
    adapted from tsai, weighted multiclass focal loss
    https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/losses.py#L116C1-L140C20
    """
    def __init__(self, 
                 weight=None, 
                 gamma=2., 
                 reduction='mean',
                 ignore_index=-100
                 ):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction
        self.ignore_index = ignore_index
    
    __name__ = 'focalloss'
        
    def forward(self, x, y):
        """
        x: [bs x n classes x n patches]
        y: [bs x n patches]
        """
        bs = x.size(0)
        
        if x.is_nested:
            max_len = max(x.size(-1) for x in x.unbind())
            x = x.to_padded_tensor(padding=0., output_size=(bs, x.size(1), max_len))
            y_output_size = (bs, max_len) if y.dim() == 2 else (bs, y.size(1), max_len)
            padding = self.ignore_index if y.dim() == 2 else 0
            y = y.to_padded_tensor(padding=padding, output_size=y_output_size)

        if x.dim() == 3:
            x = x.permute(0,2,1) # bs x n_patches x n_classes
            x = x.reshape(-1, x.size(-1)) # bs * n_patches x n_classes

        log_prob = F.log_softmax(x, dim=-1)
        weight = self.weight.float().to(x.device) if self.weight is not None else None
        if y.dim() == 2:
            # hard labels
            y = y.flatten(start_dim=-2) # bs * num_patches
            pt = log_prob[torch.arange(len(x)), y].exp()
            ce = F.nll_loss(log_prob, y, weight=weight, reduction='none', ignore_index=self.ignore_index)
            loss = (1 - pt) ** self.gamma * ce
            mask = (y != self.ignore_index).float()
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1e-5) if self.reduction == 'mean' else loss.sum()
        else:  # soft labels
            y = y.permute(0,2,1) # bs x n patches x n classes
            mask = (y.sum(dim=2) > 0).float() # bs x n patches
            mask = mask.flatten(start_dim=-2) # bs * n_patches
            y = y.reshape(-1, y.size(-1)) # bs * n_patches x n_classes

            if weight is not None:
                ce = -(y * log_prob * weight).sum(dim=-1) # bs * n_patches
            else:
                ce = -(y * log_prob).sum(dim=-1) # bs * n_patches
            pt = (y * log_prob.exp()).sum(dim=-1).clamp(min=1e-7, max=1)
            loss = (1 - pt) ** self.gamma * ce
            # Positions to ignore will have all zeros
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1e-5) if self.reduction == 'mean' else loss.sum()
        return loss

In [ ]:
#| notest
criterion = FocalLoss(gamma=0.7, weight=None, ignore_index=0)
batch_size = 10

n_patch = 721
n_class = 5
#m = torch.nn.Softmax(dim=-1)
logits = torch.randn(batch_size, n_class, n_patch)
target = torch.randint(0, n_class, size=(batch_size, n_patch))
criterion(logits, target)

tensor(8.4217)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()